# Populate testing set
This notebook is intended to populate the testing set in the linked lakehouse. 

This testing set includes the following information: 
| Object | Intent | Note |
|----------|----------|----------|
| Prompt  | Natural Language question related to the query. This question will be tested in the Data Agent  | Preferably in English |
| WorkspaceName | The name of the workspace where the source resides that contains the right data to generate an answer to the question | No workspace IDs (guids) are allowed |
| SourceType  | Semantic Model / Lakehouse  | Currently only Semantic Model and Lakehouse are supported |
| Expected Source | The name of the source in the workspace | This should be the name of the lakehouse  or semantic model containing the data to generate an answer to the question _(This field is not madatory when a SQL query is used, as the query already contains this information. However, for analytical purposes it is still helpful to save)_|
| Query | DAX or SQL expression | This field must contain the expression that generates an answer to the prompt in the related data source. The result of this query will be validated and compared to the response of the Data Agent


# Show current test set

In [12]:
df = spark.sql("SELECT * FROM LH_MONIT_DataAgentQuality.dbo.dynamic_evaluation_prompts LIMIT 1000")
display(df)

StatementMeta(, b961fe71-7cb6-49c4-938e-468dbc3b0071, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4440d5ea-b81e-4a13-a9f0-c4d8d2af9401)

# Add new test records
In case of SQL, ensure the Query is a SparkSQL Query

In [1]:
# Set table structure
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("Prompt", StringType(), True),
    StructField("WorkspaceName", StringType(), True),
    StructField("SourceType", StringType(), True),
    StructField("ExpectedSource", StringType(), True),
    StructField("Query", StringType(), True)
])

df = spark.createDataFrame([], schema)

StatementMeta(, b961fe71-7cb6-49c4-938e-468dbc3b0071, 3, Finished, Available, Finished, False)

In [13]:
# Define data to insert in testing set
data = [
    (
        "What is the price difference between retail and paid price? ",
        "FabCon ATL",
        "Semantic Model",        
        "Fuel Analysis",
        """
        EVALUATE
        ROW(
            "Average Price Difference per Liter",
            [Price per liter difference average]
        )
        """
    ),
    (
        "What is the current mileage on R-306-ZJ? ",
        "FabCon ATL",
        "Semantic Model",        
        "Fuel Analysis",
        """
        EVALUATE
        ROW(
            "Latest Mileage",
            CALCULATE(
            MAX('transactions'[Mileage]),
            'car'[License Plate] = "R-306-ZJ"
            )
        )
        """
    ),
    (
        "What is the price increase for Euro 95 between 2010 and now?",
        "FabCon ATL",
        "Lakehouse",
        "LH_Store_Gold",
        """
        WITH prices AS (
            SELECT
                `Date`,
                BenzineEuro95,
                YEAR(`Date`) AS Year
            FROM LH_Store_Gold.dbo.`recommended retail price`
            WHERE YEAR(`Date`) IN (2010, YEAR(current_date()))
        ),
        year_prices AS (
            SELECT
                Year,
                FIRST_VALUE(BenzineEuro95) OVER (
                    PARTITION BY Year ORDER BY `Date` ASC
                ) AS StartPrice,
                LAST_VALUE(BenzineEuro95) OVER (
                    PARTITION BY Year
                    ORDER BY `Date`
                    ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
                ) AS EndPrice
            FROM prices
        )
        SELECT
            MAX(CASE WHEN Year = YEAR(current_date()) THEN EndPrice END) -
            MAX(CASE WHEN Year = 2010 THEN StartPrice END) AS Price_Increase
        FROM year_prices
        """
    ),
    (
        "For how many cars did we collect data about fuel transactions?", # Prompt to test / validate
        "FabCon ATL", # WorkspaceName: Workspace where the source resides
        "Lakehouse", # SourceType: either Lakehouse or Semantic Model
        "LH_Store_Gold", # Expected source: Name of the LH / SM
        """
        SELECT COUNT(DISTINCT c.CarKey) AS NumberOfCars
        FROM LH_Store_Gold.dbo.car c
        INNER JOIN LH_Store_Gold.dbo.transactions t
            ON c.CarKey = t.CarKey
        """ # Query that generates the desired result
    )      
]

df = spark.createDataFrame(data, schema)

StatementMeta(, b961fe71-7cb6-49c4-938e-468dbc3b0071, 15, Finished, Available, Finished, False)

In [14]:
# Save data to testing set in related lakehouse
df.write.mode("overwrite").saveAsTable("dynamic_evaluation_prompts")

StatementMeta(, b961fe71-7cb6-49c4-938e-468dbc3b0071, 16, Finished, Available, Finished, False)

# Area to create and test expressions

In [4]:
df = spark.sql("""
SELECT COUNT(DISTINCT c.CarKey) AS NumberOfCars
FROM LH_Store_Gold.dbo.car c
INNER JOIN LH_Store_Gold.dbo.transactions t
    ON c.CarKey = t.CarKey
""")

display(df)

StatementMeta(, b961fe71-7cb6-49c4-938e-468dbc3b0071, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f98306e1-825b-4204-89d0-6e2dacaf9441)